# 05 Error Analysis

This notebook performs validation-only error analysis across the implemented traditional ML baselines and deep-learning model families. When available, it includes artifacts from the MSResCNN-MLP, 31-epoch MSResCNN-MLP-TCN, and 61-epoch MSResCNN-MLP-TCN models. The held-out test split is not loaded or evaluated here.

## Setup

This section imports reusable error-analysis helpers and configures paths for validation prediction artifacts. These helpers produce comparable metrics, coverage summaries, error-type counts, participant-level summaries, transition-neighborhood diagnostics, confidence diagnostics, model-agreement summaries, and confusion-matrix figures.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.error_analysis import (
    DEFAULT_STAGE13_OUTPUT_DIR,
    build_stage6_validation_prediction_tables,
    discover_validation_prediction_files,
    load_discovered_predictions,
    materialize_deep_validation_predictions_from_checkpoints,
    run_stage13_error_analysis,
)

data_interim_dir = repo_root / "data/interim"
data_processed_dir = repo_root / "data/processed"
results_dir = repo_root / "results"
stage13_output_dir = repo_root / DEFAULT_STAGE13_OUTPUT_DIR
stage13_prediction_dir = stage13_output_dir / "predictions"

stage13_output_dir.mkdir(parents=True, exist_ok=True)
stage13_prediction_dir.mkdir(parents=True, exist_ok=True)
stage13_output_dir

## Traditional ML Baseline Prediction Tables

The traditional ML baseline notebook (i.e., `notebooks/03_feature_baselines.ipynb`) saves validation metrics and confusion matrices, but not epoch-level prediction tables. This guarded section rebuilds the traditional ML baselines from the saved train/validation feature tables and writes validation prediction CSVs for error analysis. Fitting and cross-validation use only `features_train.csv`; `features_val.csv` is used only for validation prediction.

In [ ]:
BUILD_STAGE6_PREDICTIONS = False

if BUILD_STAGE6_PREDICTIONS:
    stage6_predictions = build_stage6_validation_prediction_tables(
        train_features_path=data_processed_dir / "features_train.csv",
        val_features_path=data_processed_dir / "features_val.csv",
        output_dir=stage13_prediction_dir,
        include_xgboost=True,
    )
    print(f"Saved {len(stage6_predictions)} Stage 6 prediction table(s).")
else:
    print("Skipping Stage 6 prediction rebuild. Set BUILD_STAGE6_PREDICTIONS = True when feature tables are available.")

## Optional Deep Prediction Export

Deep-learning runs from the current training utilities write `validation_epoch_predictions.csv`, `validation_aggregated_epoch_predictions_*.csv`, or ensemble validation prediction files depending on model family. For routine error analysis, leave the export flag below as `False` and first inspect the discovery table. Only enable it if a completed validation run has a saved checkpoint but is missing its validation prediction CSV; the helper performs validation inference only and records per-run errors in the summary table instead of using the test split.

In [ ]:
MATERIALIZE_DEEP_PREDICTIONS_FROM_CHECKPOINTS = False

if MATERIALIZE_DEEP_PREDICTIONS_FROM_CHECKPOINTS:
    print("Exporting missing validation predictions from saved checkpoints; no training or test evaluation is run.")
    materialization_summary = materialize_deep_validation_predictions_from_checkpoints(
        results_dir=results_dir,
        overwrite=False,
    )
    display(materialization_summary)
    if not materialization_summary.empty and "status" in materialization_summary:
        failed_exports = materialization_summary[materialization_summary["status"] == "error"]
        if not failed_exports.empty:
            print("One or more checkpoint exports failed; review the error rows before continuing.")
else:
    print("Skipping checkpoint-based deep prediction export. First inspect prediction_discovery below; set MATERIALIZE_DEEP_PREDICTIONS_FROM_CHECKPOINTS = True only for completed runs that have checkpoints but no validation prediction CSV.")

## Discover Validation Predictions

This section discovers available validation prediction tables from the traditional ML baselines, earlier deep-learning models, aggregated many-to-many predictions, and current MSResCNN-MLP, 31-epoch MSResCNN-MLP-TCN, and 61-epoch MSResCNN-MLP-TCN artifacts when available.

In [ ]:
prediction_discovery = discover_validation_prediction_files(results_dir=results_dir)
prediction_discovery

## Load Epoch Metadata

`epoch_index.csv` adds participant identity, valid validation coverage, missingness columns, and true temporal transition context. If it is unavailable, the notebook still runs the model-level analyses that only require prediction tables.

In [ ]:
epoch_index_path = data_interim_dir / "epoch_index.csv"
if epoch_index_path.exists():
    epoch_index = pd.read_csv(epoch_index_path, dtype={"participant_id": str})
    print(f"Loaded epoch index with {len(epoch_index):,} row(s).")
else:
    epoch_index = pd.DataFrame()
    print(f"Epoch index not found at {epoch_index_path}; metadata-linked summaries will be skipped or empty.")

## Run Validation Error Analysis

This cell writes validation error-analysis artifacts under `results/stage13_error_analysis/`. It is validation-only: it analyzes whatever validation prediction tables are available and does not load the held-out test feature table or test epochs.

In [ ]:
if prediction_discovery.empty:
    print("No validation error-analysis-compatible prediction files were found yet.")
    print("Run completed model stages after this implementation, or enable the Stage 6 prediction rebuild above.")
    stage13_outputs = {}
else:
    validation_predictions = load_discovered_predictions(prediction_discovery)
    stage13_outputs = run_stage13_error_analysis(
        validation_predictions,
        epoch_index=epoch_index,
        output_dir=stage13_output_dir,
        transition_radius=2,
        high_confidence_threshold=0.80,
        make_plots=True,
    )
    print(f"Analyzed {len(validation_predictions):,} validation prediction row(s).")
    print(f"Saved validation error-analysis artifacts to {stage13_output_dir.resolve()}.")
    display(stage13_outputs["model_validation_metrics"])
    display(stage13_outputs["model_coverage"])

## Error Patterns

Review the most common true-label to predicted-label mistakes, participant-level failures, temporal transition effects, and high-confidence errors. These outputs are intended to inform future work such as richer features, calibration, temporal smoothing, or simple ensembling.

In [ ]:
if stage13_outputs:
    display(stage13_outputs["error_type_summary"].head(20))
    display(stage13_outputs["participant_error_summary"].head(20))
    if "temporal_error_summary" in stage13_outputs:
        display(stage13_outputs["temporal_error_summary"])
    display(stage13_outputs["confidence_summary"])
    display(stage13_outputs["model_disagreement_summary"])
    display(stage13_outputs["shared_epoch_model_metrics"])
else:
    print("Run the validation error-analysis cell above after prediction tables are available.")

## Error Plots

This section creates compact validation-only figures from the saved error-analysis tables. Figures are written to `results/stage13_error_analysis/presentation_figures/` so the original analysis tables and confusion-matrix figures remain unchanged.

In [ ]:
presentation_figure_dir = stage13_output_dir / "presentation_figures"
presentation_figure_dir.mkdir(parents=True, exist_ok=True)

SELECTED_MODEL_LABELS = {
    "majority_class": "Majority",
    "logistic_elasticnet": "Logistic regression",
    "xgboost_all_features": "XGBoost",
    "stage9_best_single_epoch_cnn": "Stage 9 CNN",
    "stage14_multiscale_fusion_cnn_sqrt_weighted": "Stage 14 fusion",
    "stage15_equal_weight_seed_ensemble": "Stage 15 ensemble",
    "stage16_equal_weight_seed_ensemble": "Stage 16 ensemble",
}
SELECTED_MODEL_ORDER = list(SELECTED_MODEL_LABELS)
STAGE16_MODEL_NAME = "stage16_equal_weight_seed_ensemble"
CLASS_LABELS = ["Wake", "Non-REM", "REM"]

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120


def load_stage13_table(name: str) -> pd.DataFrame:
    outputs = globals().get("stage13_outputs", {})
    if isinstance(outputs, dict) and name in outputs:
        return outputs[name].copy()
    path = stage13_output_dir / f"{name}.csv"
    if path.exists():
        return pd.read_csv(path, dtype={"participant_id": str})
    return pd.DataFrame()


def selected_model_frame(metrics: pd.DataFrame) -> pd.DataFrame:
    selected = metrics[metrics["model_name"].isin(SELECTED_MODEL_ORDER)].copy()
    selected["model_label"] = selected["model_name"].map(SELECTED_MODEL_LABELS)
    selected["model_order"] = selected["model_name"].map(
        {name: index for index, name in enumerate(SELECTED_MODEL_ORDER)}
    )
    return selected.sort_values("model_order")


def save_presentation_figure(fig, filename: str) -> Path:
    path = presentation_figure_dir / filename
    fig.savefig(path, bbox_inches="tight", dpi=200)
    print(f"Saved {path}")
    return path


presentation_figure_dir

### Model-Level Validation Performance

This figure compares validation macro F1 across the main baseline and deep-learning models. The artifact is `presentation_figures/model_macro_f1_summary.png`.

In [ ]:
metrics = load_stage13_table("model_validation_metrics")
selected_metrics = selected_model_frame(metrics)

if selected_metrics.empty:
    print("Model validation metrics are unavailable; run the analysis cell first.")
else:
    plot_frame = selected_metrics.sort_values("macro_f1")
    colors = [
        "#2f6f9f" if name == STAGE16_MODEL_NAME else "#9fb7c9"
        for name in plot_frame["model_name"]
    ]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh(plot_frame["model_label"], plot_frame["macro_f1"], color=colors)
    ax.set_xlabel("Validation macro F1")
    ax.set_ylabel("")
    ax.set_xlim(0, max(0.55, float(plot_frame["macro_f1"].max()) + 0.04))
    ax.set_title("Validation macro F1 by model")
    for value, label in zip(plot_frame["macro_f1"], plot_frame["model_label"]):
        ax.text(value + 0.008, label, f"{value:.3f}", va="center", fontsize=11)
    fig.tight_layout()
    save_presentation_figure(fig, "model_macro_f1_summary.png")
    plt.show()

### Class-Level Behavior

This figure shows Wake, Non-REM, and REM F1 for the selected comparison models. The artifact is `presentation_figures/selected_model_class_f1.png`.

In [ ]:
if selected_metrics.empty:
    print("Model validation metrics are unavailable; run the analysis cell first.")
else:
    class_f1 = selected_metrics[
        ["model_label", "Wake_f1", "Non_REM_f1", "REM_f1"]
    ].rename(
        columns={"Wake_f1": "Wake", "Non_REM_f1": "Non-REM", "REM_f1": "REM"}
    )
    class_f1 = class_f1.set_index("model_label")[CLASS_LABELS]
    fig, ax = plt.subplots(figsize=(11, 5.5))
    class_f1.plot(kind="bar", ax=ax, color=["#4c78a8", "#59a14f", "#e15759"])
    ax.set_xlabel("")
    ax.set_ylabel("Validation F1")
    ax.set_ylim(0, 0.9)
    ax.set_title("Per-class validation F1")
    ax.legend(title="Class", frameon=False, ncol=3)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    save_presentation_figure(fig, "selected_model_class_f1.png")
    plt.show()

### 61-Epoch MSResCNN-MLP-TCN Confusion Patterns

This figure normalizes the 3-Seed 61-Epoch MSResCNN-MLP-TCN ensemble confusion matrix by true class, so each row shows the percentage of that true class assigned to each predicted class. The artifact is `presentation_figures/stage16_normalized_confusion_matrix.png`.

In [ ]:
combined_predictions = load_stage13_table("combined_validation_predictions")
stage16_predictions = combined_predictions[
    combined_predictions.get("model_name", pd.Series(dtype="object")) == STAGE16_MODEL_NAME
].copy()

if stage16_predictions.empty:
    print("Stage 16 ensemble predictions are unavailable; run the analysis cell first.")
else:
    counts = pd.crosstab(
        stage16_predictions["true_label"],
        stage16_predictions["pred_label"],
    ).reindex(index=CLASS_LABELS, columns=CLASS_LABELS, fill_value=0)
    normalized = counts.div(counts.sum(axis=1), axis=0)
    annotations = normalized.apply(lambda column: column.map(lambda value: f"{value:.0%}"))

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    sns.heatmap(
        normalized,
        annot=annotations,
        fmt="",
        cmap="Blues",
        vmin=0,
        vmax=1,
        cbar_kws={"label": "Row percentage"},
        ax=ax,
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Stage 16 ensemble normalized confusion matrix")
    fig.tight_layout()
    save_presentation_figure(fig, "stage16_normalized_confusion_matrix.png")
    plt.show()
    display(counts)

### Transition-Neighborhood Effects

This figure compares the 3-Seed 61-Epoch MSResCNN-MLP-TCN ensemble performance away from transitions versus within the configured transition neighborhood. The artifact is `presentation_figures/stage16_transition_effects.png`.

In [ ]:
temporal = load_stage13_table("temporal_error_summary")
stage16_temporal = temporal[temporal.get("model_name", pd.Series(dtype="object")) == STAGE16_MODEL_NAME].copy()

if stage16_temporal.empty:
    print("Stage 16 temporal summary is unavailable; run the analysis cell first.")
else:
    stage16_temporal["region"] = np.where(
        stage16_temporal["near_transition"].astype(str).str.lower() == "true",
        "Near transition",
        "Stable region",
    )
    transition_plot = stage16_temporal.set_index("region")[
        ["accuracy", "macro_f1", "REM_recall"]
    ].rename(columns={"accuracy": "Accuracy", "macro_f1": "Macro F1", "REM_recall": "REM recall"})
    transition_plot = transition_plot.reindex(["Stable region", "Near transition"])

    fig, ax = plt.subplots(figsize=(8, 5.5))
    transition_plot.plot(kind="bar", ax=ax, color=["#4c78a8", "#f28e2b", "#e15759"])
    ax.set_xlabel("")
    ax.set_ylabel("Validation metric")
    ax.set_ylim(0, 0.8)
    ax.set_title("Stage 16 performance near sleep-stage transitions")
    ax.legend(frameon=False, ncol=3)
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    save_presentation_figure(fig, "stage16_transition_effects.png")
    plt.show()

### Participant-Level Variability

This figure shows the 3-Seed 61-Epoch MSResCNN-MLP-TCN ensemble macro F1 for each validation participant. The artifact is `presentation_figures/stage16_participant_macro_f1.png`.

In [ ]:
participant_summary = load_stage13_table("participant_error_summary")
stage16_participants = participant_summary[
    participant_summary.get("model_name", pd.Series(dtype="object")) == STAGE16_MODEL_NAME
].copy()

if stage16_participants.empty:
    print("Stage 16 participant summary is unavailable; run the analysis cell first.")
else:
    stage16_participants = stage16_participants.sort_values("macro_f1")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        stage16_participants["macro_f1"],
        stage16_participants["participant_id"],
        s=70,
        color="#2f6f9f",
    )
    ax.axvline(stage16_participants["macro_f1"].median(), color="#555555", linestyle="--", linewidth=1)
    ax.set_xlabel("Validation macro F1")
    ax.set_ylabel("Participant")
    ax.set_xlim(0, 0.8)
    ax.set_title("Stage 16 participant-level variability")
    fig.tight_layout()
    save_presentation_figure(fig, "stage16_participant_macro_f1.png")
    plt.show()

### Confidence Diagnostics

This figure compares the 3-Seed 61-Epoch MSResCNN-MLP-TCN ensemble confidence-bin accuracy against mean confidence. The artifact is `presentation_figures/stage16_confidence_bins.png`.

In [ ]:
confidence_bins = load_stage13_table("confidence_bins")
stage16_bins = confidence_bins[
    confidence_bins.get("model_name", pd.Series(dtype="object")) == STAGE16_MODEL_NAME
].copy()

if stage16_bins.empty:
    print("Stage 16 confidence bins are unavailable; run the analysis cell first.")
else:
    for column in ["n_predictions", "accuracy", "mean_confidence"]:
        stage16_bins[column] = pd.to_numeric(stage16_bins[column], errors="coerce")
    stage16_bins = stage16_bins.dropna(subset=["n_predictions", "accuracy", "mean_confidence"])
    stage16_bins = stage16_bins[stage16_bins["n_predictions"] > 0]

    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.plot([0, 1], [0, 1], color="#777777", linestyle="--", linewidth=1, label="Perfect calibration")
    ax.scatter(
        stage16_bins["mean_confidence"],
        stage16_bins["accuracy"],
        s=np.sqrt(stage16_bins["n_predictions"]) * 12,
        color="#2f6f9f",
        alpha=0.85,
    )
    for _, row in stage16_bins.iterrows():
        ax.text(row["mean_confidence"] + 0.01, row["accuracy"], str(row["confidence_bin"]), fontsize=9)
    ax.set_xlabel("Mean confidence")
    ax.set_ylabel("Accuracy")
    ax.set_xlim(0.3, 0.95)
    ax.set_ylim(0.3, 0.95)
    ax.set_title("Stage 16 confidence-bin accuracy")
    ax.legend(frameon=False)
    fig.tight_layout()
    save_presentation_figure(fig, "stage16_confidence_bins.png")
    plt.show()

### High-Confidence Error Types

This figure summarizes the most common 3-Seed 61-Epoch MSResCNN-MLP-TCN ensemble errors among predictions above the configured high-confidence threshold. The artifact is `presentation_figures/stage16_high_confidence_error_types.png`.

In [ ]:
high_confidence_errors = load_stage13_table("high_confidence_errors")
stage16_high_confidence = high_confidence_errors[
    high_confidence_errors.get("model_name", pd.Series(dtype="object")) == STAGE16_MODEL_NAME
].copy()

if stage16_high_confidence.empty:
    print("No Stage 16 high-confidence errors are available at the configured threshold.")
else:
    error_counts = stage16_high_confidence["error_type"].value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.barh(error_counts.index, error_counts.values, color="#d96b5f")
    ax.set_xlabel("High-confidence error count")
    ax.set_ylabel("")
    ax.set_title("Stage 16 high-confidence error types")
    for count, label in zip(error_counts.values, error_counts.index):
        ax.text(count + 2, label, f"{count:,}", va="center", fontsize=11)
    fig.tight_layout()
    save_presentation_figure(fig, "stage16_high_confidence_error_types.png")
    plt.show()

### Validation Summary Panel

This four-panel figure combines the model-level comparison, the 3-Seed 61-Epoch MSResCNN-MLP-TCN per-class F1, transition-neighborhood effect, and participant-level variability. The artifact is `presentation_figures/validation_summary_panel.png`.

In [ ]:
required_tables_available = (
    not selected_metrics.empty
    and not stage16_participants.empty
    and not stage16_temporal.empty
)

if not required_tables_available:
    print("Summary panel inputs are unavailable; run the preceding analysis and figure cells first.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    ax = axes[0, 0]
    compact_models = selected_metrics.sort_values("macro_f1")
    colors = [
        "#2f6f9f" if name == STAGE16_MODEL_NAME else "#9fb7c9"
        for name in compact_models["model_name"]
    ]
    ax.barh(compact_models["model_label"], compact_models["macro_f1"], color=colors)
    ax.set_xlabel("Macro F1")
    ax.set_title("Model comparison")

    ax = axes[0, 1]
    stage16_row = selected_metrics[selected_metrics["model_name"] == STAGE16_MODEL_NAME].iloc[0]
    class_values = [stage16_row["Wake_f1"], stage16_row["Non_REM_f1"], stage16_row["REM_f1"]]
    ax.bar(CLASS_LABELS, class_values, color=["#4c78a8", "#59a14f", "#e15759"])
    ax.set_ylim(0, 0.9)
    ax.set_ylabel("F1")
    ax.set_title("Stage 16 per-class F1")

    ax = axes[1, 0]
    transition_plot[["Macro F1", "REM recall"]].plot(kind="bar", ax=ax, color=["#f28e2b", "#e15759"])
    ax.set_ylim(0, 0.65)
    ax.set_xlabel("")
    ax.set_ylabel("Metric")
    ax.set_title("Transition effect")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(frameon=False)

    ax = axes[1, 1]
    ax.boxplot(stage16_participants["macro_f1"], vert=False, widths=0.35)
    ax.scatter(stage16_participants["macro_f1"], np.ones(len(stage16_participants)), alpha=0.8, color="#2f6f9f")
    ax.set_yticks([])
    ax.set_xlim(0, 0.8)
    ax.set_xlabel("Participant macro F1")
    ax.set_title("Participant variability")

    fig.suptitle("Validation error-analysis summary", y=1.02, fontsize=18)
    fig.tight_layout()
    save_presentation_figure(fig, "validation_summary_panel.png")
    plt.show()